# Zipf profile for one corpus

Rank-frequency (Zipf) curves for a single system, across the `raw` / `basic` / `cleansed` name tiers.

Every computation and every curve comes from `analysis.token_zipf`, the same module `scripts/analyze_token_zipf.py` uses for its batch runs. This notebook only selects a corpus, calls that module, and displays the result: if something here grows into logic worth trusting, it belongs in `src/analysis/token_zipf.py` where it can be tested and reused, not in this file.

For all systems at once, with persisted artifacts and a written report, use the script instead:

```bash
uv run python scripts/analyze_token_zipf.py --systems gb
```

In [ ]:
# --- parameters -------------------------------------------------------------
SYSTEM = "ie"  # fr | gb | gleif | ie | offeneregister | wikidata
TIERS = ["raw", "basic", "cleansed"]
WORDFREQ_TOP_N = 5000
# ----------------------------------------------------------------------------

import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import _bootstrap  # noqa: F401

from workspace.roots import default_workspace_roots

ROOTS = default_workspace_roots(ROOT)

import matplotlib.pyplot as plt
import polars as pl

from analysis.token_rarity import resolve_system_language
from analysis.token_zipf import (
    NAME_TIER_COLUMNS,
    build_wordfreq_reference_curve,
    compare_power_law_and_lognormal,
    discover_cleansed_files,
    draw_head_words,
    draw_zipf_curve,
    load_persisted_power_law_fit,
    load_persisted_word_token_stats,
)

# The script renders to PNG under artifacts/; here we draw onto our own axes so
# figures appear inline. `matplotlib.use("Agg")` runs at token_zipf import time,
# so switch to the notebook backend after importing it.
%matplotlib inline

files = discover_cleansed_files(ROOTS, SYSTEM)
language = resolve_system_language(SYSTEM)
print(f"{SYSTEM}: {len(files)} cleansed shard(s), wordfreq language={language!r}")
if not files:
    raise FileNotFoundError(
        f"No cleansed shards for {SYSTEM!r} under {ROOTS.data / SYSTEM / 'cleansed'}."
    )

## Token statistics per tier

`compute_word_token_stats` (in `src/analysis/token_zipf.py`) counts **document frequency**: the number of names containing a word at least once, read directly from the cleansed shards' text -- not tokenizer output, so results are unaffected by the tokenize-stage noise-word trim. That's a full pass over the corpus per tier, expensive enough (three-figure MB, millions of names) that this notebook never runs it itself.

Instead, `scripts/analyze_token_zipf.py` runs it once and persists the result under `artifacts/analysis/token_zipf/runs/<run_date>/metrics/`; this cell only reads that back via `load_persisted_word_token_stats`. A tier with no persisted run is skipped with the command to generate it, not silently computed inline -- re-run that command for `SYSTEM` after a cleanse/tokenize change to refresh stale numbers.

A tier is also skipped when the corpus has no column for it.

In [3]:
persisted = load_persisted_word_token_stats(ROOTS, SYSTEM)

available_columns = set(pl.read_parquet_schema(files[0]))

stats_by_tier: dict[str, tuple[pl.DataFrame, int]] = {}
for tier in TIERS:
    name_col = NAME_TIER_COLUMNS[tier]
    if name_col not in available_columns:
        print(f"skipping {tier!r}: no {name_col!r} column in {SYSTEM}")
        continue
    if persisted is None or tier not in persisted:
        print(
            f"skipping {tier!r}: not calculated yet -- run "
            f"`uv run --no-sync python scripts/analyze_token_zipf.py --systems {SYSTEM}` "
            "to generate it, then re-run this cell"
        )
        continue
    stats, total_docs = persisted[tier]
    stats_by_tier[tier] = (stats, total_docs)
    print(f"{tier:9} {name_col:22} {total_docs:>12,} names  {stats.height:>10,} words")

if not stats_by_tier:
    raise RuntimeError(
        f"No persisted token stats for {SYSTEM!r} in any tier; nothing below can "
        "run. Generate them with the command printed above, then re-run this cell."
    )

raw       name                        817,761 names     296,095 words
basic     name_cleansed_basic         817,761 names     297,673 words
cleansed  name_cleansed               817,761 names     297,661 words


## Zipf curves

Each corpus curve is drawn against a general-language reference from `wordfreq`. Both are normalized to their own rank-1 value, so what is being compared is the shape of the decay, not absolute scale: corpus document frequency and wordfreq's per-billion-word scale are different units.

The fitted slope is reported per curve. A corpus decaying much faster or slower than general language is the divergence this is here to show.

In [ ]:
reference = (
    build_wordfreq_reference_curve(language, top_n=WORDFREQ_TOP_N) if language else None
)
if reference is None:
    print(f"No wordfreq reference language for {SYSTEM!r}; plotting the corpus alone.")

fig, axes = plt.subplots(
    1, len(stats_by_tier), figsize=(7 * len(stats_by_tier), 6), squeeze=False
)

slopes = []
for ax, (tier, (stats, total_docs)) in zip(axes[0], stats_by_tier.items()):
    result = draw_zipf_curve(
        ax,
        stats,
        reference,
        title=f"{SYSTEM} / {tier} ({total_docs:,} names)",
    )
    slopes.append({"tier": tier, **result})

fig.tight_layout()
plt.show()

In [ ]:
# Show the slope data
pl.DataFrame(slopes)

## Power law vs log-normal

All of the company name corpora have a number of hyper-dominant words, so it is useful to discard those and also examine the tail distribution as visually distinguising a power-law from a log-normal distribution is .
We apply the following tests:
* Power-Law: `powerlaw` (Clauset-Shalizi-Newman) and a Kolmogorov-Smirnov statistic for power-law fit's own goodness of fit (`ks_statistic`), lower is a tighter fit.
* Log-likihood ratio: `loglikelihood_ratio` (Vuong's test) comparing power-law against log-normal as competing models  positive favors power-law, negative favors log-normal; `p_value` is that preference's significance (above roughly 0.1 the test can't tell the two apart).
* `mean_loglikelihood_diff`: the same comparison as a per-token average, independent of corpus size -- the number to compare across systems/tiers, since `loglikelihood_ratio`/`p_value` grow with n for a fixed effect and aren't comparable across corpora of different sizes.

`IGNORE_TOP_N_WORDS` drops that many of the highest-document-frequency words before fitting. Leave it `None` to fit the whole vocabulary, which is what the batch run persists: `powerlaw` estimates its own lower cutoff `xmin`, and a spike of one or two legal-form words lies far above any cutoff it picks. Set an integer to see whether taking the head off by hand moves the result; the most-frequent-words picture further down says how many words the head holds.

**Persisted vs computed:** with `IGNORE_TOP_N_WORDS = None`, this cell first looks for a result already computed by `scripts/analyze_token_zipf.py --power-law-compare` and reuses it instantly if found (labeled `[persisted]` below) -- only computing live (`[computed]`) when there's nothing to reuse. Setting `IGNORE_TOP_N_WORDS` to an explicit int always computes live at that cutoff, since a persisted result computed at a different cutoff wouldn't answer the question you're asking.

**Runtime:** discrete `xmin` estimation is expensive, on offeneregister's ~1M-token cleansed-tier vocabulary this takes ~70s.

In [ ]:
# --- power-law vs log-normal fit --------------------------------------
POWERLAW_TIER = "cleansed"  # one of TIERS
# None = reuse the persisted fit (from a `--power-law-compare` batch run) if
# one exists, falling back to computing it live over the whole vocabulary.
# Set an int to always compute live with that many head words dropped.
IGNORE_TOP_N_WORDS = None
# --------------------------------------------------------------------------

if POWERLAW_TIER not in stats_by_tier:
    raise ValueError(
        f"POWERLAW_TIER={POWERLAW_TIER!r} has no stats; choose one of "
        f"{sorted(stats_by_tier)}"
    )

fit_result = None
if IGNORE_TOP_N_WORDS is None:
    fit_result = load_persisted_power_law_fit(ROOTS, SYSTEM, POWERLAW_TIER)

skip_top_n = IGNORE_TOP_N_WORDS or 0
if fit_result is not None:
    source = "persisted"
else:
    stats, total_docs = stats_by_tier[POWERLAW_TIER]
    fit_result = compare_power_law_and_lognormal(stats, skip_top_n=skip_top_n)
    source = "computed"

print(
    f"{SYSTEM} / {POWERLAW_TIER} [{source}]: ignoring top {skip_top_n} words, "
    f"{int(fit_result['n_fitted']):,} tokens fit"
)
pl.DataFrame([fit_result])

## Most frequent words

The head of each tier's distribution as named words: the legal-form spike, the shelf of descriptor words under it, and the rank at which the shelf gives way to the tail, which the rank-frequency curve compresses into its first decade. Drawn by `draw_head_words`, the same function the batch run writes `<system>_<tier>_head_words.png` with. Pick the tier for the table and the word count for both with `TOP_WORDS_TIER` / `TOP_WORDS_N` at the top of the cell below. Legal-form words dominating the cleansed tier is expected: the cleanse stage extracts the company type into its own column but does not remove every such word from the name.

In [ ]:
# --- pick which table to show ------------------------------------------
TOP_WORDS_TIER = "cleansed"  # one of TIERS
TOP_WORDS_N = 30  # how many words to draw and rows to show
# --------------------------------------------------------------------------

if TOP_WORDS_TIER not in stats_by_tier:
    raise ValueError(
        f"TOP_WORDS_TIER={TOP_WORDS_TIER!r} has no stats; choose one of "
        f"{sorted(stats_by_tier)}"
    )

fig, axes = plt.subplots(
    len(stats_by_tier), 1, figsize=(10, 4.5 * len(stats_by_tier)), squeeze=False
)
for ax, (tier, (stats, total_docs)) in zip(axes[:, 0], stats_by_tier.items()):
    draw_head_words(
        ax,
        stats,
        title=f"{SYSTEM} / {tier} -- {TOP_WORDS_N} most frequent words",
        top_n=TOP_WORDS_N,
    )
fig.tight_layout()
plt.show()

stats, total_docs = stats_by_tier[TOP_WORDS_TIER]
top = (
    stats.sort("document_frequency", descending=True)
    .head(TOP_WORDS_N)
    .with_columns(
        (pl.col("document_frequency") / total_docs * 100).round(2).alias("pct_of_names")
    )
)
print(f"{SYSTEM} / {TOP_WORDS_TIER} (top {TOP_WORDS_N} of {stats.height:,} words)")
top.to_pandas()